# Introduction to AI Agents — Solved Notebook with Hugging Face APIs

This notebook implements the exercises using the **Hugging Face Inference API**.

We use two ideas:

1. A normal LLM call.
2. A ReAct-style agent loop: **Thought → Action → Observation → Thought → Action → ...**

> **Important:** A reasoning model is **not required** to build an agent. A capable instruction-tuned model can decide which tool to call and can follow the ReAct format. Reasoning models can improve difficult planning/reasoning tasks, but they are optional.

Set your Hugging Face token in the environment variable `HF_TOKEN` before running the API cells.

## 1. A simple LLM call vs. an agent

A single LLM call is:

```text
User → LLM → Answer
```

An agent is a loop:

```text
Goal → LLM → Tool → Observation → LLM → Tool → Observation → Answer
```

The important difference is **interaction with an external environment**. The agent can obtain information that was not present in the original prompt.

In [1]:
import os
from huggingface_hub import InferenceClient

HF_TOKEN = os.getenv("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN is not set. Set it as an environment variable before running this cell."
    )

# You can change this to another Hugging Face Inference Provider model.
MODEL = "Qwen/Qwen2.5-7B-Instruct"

client = InferenceClient(
    api_key=HF_TOKEN
)

print("Client ready.")
print("Model:", MODEL)

/home/neeraj/Projects/genai_fundamentals/genai_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Client ready.
Model: Qwen/Qwen2.5-7B-Instruct


In [2]:
def hf_llm(messages, temperature=0.2, max_tokens=300):
    """Small wrapper around Hugging Face chat completion."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
    )
    return response.choices[0].message.content


def single_llm_call(user_request: str) -> str:
    """A single LLM call: one input, one output, done."""
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer concisely."
        },
        {
            "role": "user",
            "content": user_request
        }
    ]
    return hf_llm(messages)


print(single_llm_call(
    "Explain in one sentence why an AI agent needs tools."
))

An AI agent needs tools to perform tasks, process information, and interact with its environment effectively.


Notice the single call above can only ever *guess* -- it has no way to
actually check flight prices. This is exactly the gap the agent loop below
is designed to close.


## 2. The agent loop: Perceive → Reason → Act

```
        +-------------------------------------+
        |                                     |
        v                                     |
   [PERCEIVE]  ---->  [REASON]  ---->  [ACT]  -+
   (read current       (LLM decides    (call a tool,
    state / last          what to do    take an action)
    tool result)           next)
```

Let's build this loop for a tiny toy task: an agent that needs to find the
cheapest of several flight options. "Perceive" starts as the user's request;
each "act" step calls a (fake) tool; the tool's result becomes the next
thing to "perceive".


In [3]:
# Deterministic fake flight data.
# Deterministic data makes the notebook easier to reproduce.

FLIGHTS = [
    {"airline": "IndiGo", "price": 4200},
    {"airline": "Air India", "price": 5100},
    {"airline": "Vistara", "price": 3900},
    {"airline": "SpiceJet", "price": 4500},
]

# Seat availability for Exercise 1.1
SEAT_AVAILABLE = {
    "Vistara": False,
    "IndiGo": True,
    "SpiceJet": True,
    "Air India": True,
}


def search_flights_tool(origin: str, destination: str) -> list:
    """Fake flight-search API."""
    return FLIGHTS.copy()


def check_seat_availability(airline: str, price: int) -> bool:
    """Fake seat-availability API."""
    return SEAT_AVAILABLE.get(airline, False)


def sort_flights_by_price(flight_options: list) -> list:
    return sorted(flight_options, key=lambda x: x["price"])

In [ ]:
def reason_pick_cheapest(flight_options: list) -> dict:
    """REASON step.

    For the basic exercise we use Python for the deterministic selection.
    Later, the Hugging Face model will perform the ReAct decision-making.
    """
    if not flight_options:
        return None
    return min(flight_options, key=lambda x: x["price"])


def run_agent(origin: str, destination: str):
    """Solved Exercise 1.1:
    Search flights → choose cheapest → check seats.
    If unavailable, try the next cheapest flight.
    """
    print(f"[PERCEIVE] Goal: find cheapest available flight {origin} -> {destination}")

    # ACT 1
    print("[ACT] search_flights_tool(...)")
    flight_options = search_flights_tool(origin, destination)
    print("[PERCEIVE] Observed:", flight_options)

    if not flight_options:
        print("[FINAL] No flights available.")
        return None

    # Try candidates in increasing price order.
    candidates = sort_flights_by_price(flight_options)

    for flight in candidates:
        # REASON
        print(f"[REASON] Considering {flight['airline']} at Rs.{flight['price']}")

        # ACT 2
        available = check_seat_availability(
            flight["airline"],
            flight["price"]
        )

        # PERCEIVE
        print(
            f"[PERCEIVE] Seat availability for {flight['airline']}: "
            f"{available}"
        )

        if available:
            print(
                f"[FINAL] Book {flight['airline']} at Rs.{flight['price']}"
            )
            return flight

        print("[REASON] This flight is unavailable; try the next cheapest option.")

    print("[FINAL] No available flight found.")
    return None


best = run_agent("Bengaluru", "Mumbai")

### Exercise 1.1 — Solution

The important part is the **loop**:

```text
search flights
     ↓
sort by price
     ↓
check cheapest
     ↓
available? ── yes → final answer
     │
     no
     ↓
try next cheapest
```

This is already an agent-like loop because the result of one action changes what the system does next.

## 3. ReAct with a Hugging Face LLM

Now we move the **decision-making** from hard-coded Python logic to the LLM.

The model is asked to produce a structured ReAct step:

```text
Thought: ...
Action: search_flights
Action Input: {...}
```

The Python program executes the action and adds:

```text
Observation: ...
```

Then the model gets another turn.

This is the key idea from the ReAct paper: **reasoning and acting are interleaved**.

In [ ]:
import json
import re


TOOLS_DESCRIPTION = """
Available tools:

1. search_flights
   Input JSON:
   {"origin": "Bengaluru", "destination": "Mumbai"}

2. check_seat_availability
   Input JSON:
   {"airline": "Vistara", "price": 3900}
"""


REACT_SYSTEM_PROMPT = f"""
You are an AI agent following the ReAct pattern.

Your job is to find the cheapest AVAILABLE flight.

You have access to these tools:

{TOOLS_DESCRIPTION}

At every step, output exactly ONE of these formats:

Thought: <brief reasoning about what information is needed next>
Action: <tool name>
Action Input: <valid JSON>

OR, when you have enough information:

Thought: <brief reasoning>
Final Answer: <answer for the user>

Rules:
- Do not invent tool results.
- Use a tool when you need information from the environment.
- If the cheapest flight has no seat, check the next cheapest flight.
- Keep the Thought brief. Do not expose long private chain-of-thought.
"""


def parse_react_output(text: str):
    """Extract one action or a final answer from the model output."""

    final_match = re.search(
        r"Final Answer:\s*(.*)",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )
    if final_match:
        return {
            "type": "final",
            "answer": final_match.group(1).strip()
        }

    action_match = re.search(
        r"Action:\s*([A-Za-z_][A-Za-z0-9_]*)",
        text,
        flags=re.IGNORECASE
    )

    input_match = re.search(
        r"Action Input:\s*(\{.*?\})",
        text,
        flags=re.IGNORECASE | re.DOTALL
    )

    if action_match and input_match:
        return {
            "type": "action",
            "action": action_match.group(1).strip(),
            "input": json.loads(input_match.group(1))
        }

    raise ValueError(
        "Could not parse the model output as a ReAct action/final answer.\n\n"
        + text
    )


def execute_tool(name: str, arguments: dict):
    if name == "search_flights":
        return search_flights_tool(
            arguments["origin"],
            arguments["destination"]
        )

    if name == "check_seat_availability":
        return check_seat_availability(
            arguments["airline"],
            arguments["price"]
        )

    raise ValueError(f"Unknown tool: {name}")

In [ ]:
def react_agent_hf(origin: str, destination: str, max_steps: int = 6):
    """A simple ReAct loop using a Hugging Face chat model."""

    messages = [
        {
            "role": "system",
            "content": REACT_SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": (
                f"Find the cheapest available flight from "
                f"{origin} to {destination}."
            )
        }
    ]

    transcript = []

    for step in range(max_steps):
        response = hf_llm(
            messages,
            temperature=0.0,
            max_tokens=300
        )

        print(f"\n--- STEP {step + 1} ---")
        print(response)

        transcript.append(response)

        parsed = parse_react_output(response)

        if parsed["type"] == "final":
            return transcript

        # Add the model's action message to the conversation.
        messages.append({
            "role": "assistant",
            "content": response
        })

        # Execute the selected tool.
        observation = execute_tool(
            parsed["action"],
            parsed["input"]
        )

        print("Observation:", observation)

        # Give the result back to the model.
        messages.append({
            "role": "user",
            "content": f"Observation: {json.dumps(observation)}"
        })

    raise RuntimeError("Agent stopped because max_steps was reached.")


transcript = react_agent_hf("Bengaluru", "Mumbai")

### Exercise 1.2 — Solution

If the tool returns an empty flight list, the agent must not invent an answer.

A robust agent should produce something equivalent to:

```text
Thought: The flight search returned no options, so I cannot select a flight.
Final Answer: No flights were found for this route.
```

The important ReAct principle is:

**Observation changes the next reasoning/action step.**

In [ ]:
# Exercise 1.2 demonstration: empty result handling

def search_flights_empty_tool(origin: str, destination: str) -> list:
    return []


empty_flights = search_flights_empty_tool("Bengaluru", "Mumbai")

if not empty_flights:
    print("Observation:", empty_flights)
    print(
        "Thought: The tool returned no flights, so I should not invent a flight."
    )
    print("Final Answer: No flights were found for this route.")

### Exercise 1.3 — Solution

`run_agent()` and `react_agent_hf()` have the same high-level structure:

```text
Perceive → Reason → Act → Perceive → ...
```

The important difference is **where the reasoning comes from**.

- `run_agent()` uses hard-coded Python rules.
- `react_agent_hf()` asks the Hugging Face LLM to decide the next action.

The visible ReAct transcript is useful for debugging because you can inspect:

- which tool the model selected,
- what arguments it generated,
- what observation it received,
- whether it reacted correctly to that observation.

## 4. Exercise 1.4 — When NOT to use an agent

### 1. Translate a sentence from English to Hindi
**Answer: (a) Single LLM call**

There is no need for external information or iterative decision-making.

### 2. "What's on my calendar tomorrow?"
**Answer: (b) Fixed multi-step workflow**

The workflow can simply be:

```text
Get current date → call calendar API → summarize events
```

A full agent loop is unnecessary unless the user asks for more dynamic actions.

### 3. "Plan and book my entire 5-day trip to Goa, adjusting for weather and budget as you go, and re-book anything that falls through."
**Answer: (c) Full agent loop**

This requires repeated observation, planning, tool calls, handling failures, and adapting to new information.

### 4. Classify a support ticket as "billing", "technical", or "other".
**Answer: (a) Single LLM call**

This is normally a straightforward classification task with no need for iterative tool use.

# 5. Does ReAct require a reasoning model?

**No.**

This distinction is important.

### ReAct is a prompting/agent architecture

The original ReAct idea is:

```text
Thought → Action → Observation → Thought → Action → ...
```

It does **not** say that the underlying model must be a dedicated reasoning model.

An instruction-tuned model can be used if it can reliably follow the requested format and choose appropriate actions.

### Reasoning models are optional

A reasoning model can be useful when:

- the task requires difficult multi-step planning,
- there are many possible tool calls,
- the agent must compare alternatives,
- the task requires substantial reasoning before acting.

But for this teaching notebook, an instruction-tuned model is actually useful because it makes the ReAct mechanism easier to understand.

### For this notebook

Use:

```python
MODEL = "Qwen/Qwen2.5-7B-Instruct"
```

and teach the architecture first.

Then optionally replace it with a reasoning model and compare behavior.

The key learning objective is **not "use a reasoning model"**. It is:

> **Use an LLM to decide what action to take, execute the action in the environment, feed the observation back to the LLM, and continue until the task is complete.**

# 6. ReAct vs modern tool calling

There are two closely related ways to teach agents.

### Paper-style ReAct

```text
LLM
 ↓
Thought
 ↓
Action text
 ↓
Python parses action
 ↓
Tool
 ↓
Observation
 ↓
LLM
```

This is close to the original ReAct paper and is excellent for understanding the concept.

### Modern function/tool calling

The model directly returns a structured tool call:

```text
LLM
 ↓
tool_call(name="search_flights", arguments={...})
 ↓
Python executes tool
 ↓
tool result
 ↓
LLM
```

Hugging Face's current Inference Providers API supports function/tool calling and `tool_choice` options such as `auto` and `required`.

For a production agent, **structured tool calling is preferable to parsing free-form `Action:` text**.

## Key takeaway

Do not confuse these three concepts:

| Concept | Meaning |
|---|---|
| LLM | Generates language/output |
| Reasoning model | A model optimized/trained for harder reasoning |
| Agent | An LLM-driven system that can repeatedly observe, decide, act, and use tool results |

Therefore:

**Agent ≠ reasoning model.**

A reasoning model can be the brain inside an agent, but it is not a requirement for an agent architecture.